# 06 · Avaliação e implantação

**CRISP-DM fases 5 e 6**

A fase 4 perguntou se o modelo é tecnicamente bom. Esta pergunta se ele **resolve o
problema que a diretoria trouxe** — o que é outra coisa.

> É aqui, e só aqui, que o rótulo reservado entra.

In [ ]:
import sys, warnings
from pathlib import Path

# O notebook roda a partir de `notebooks/`; a raiz do projeto entra no path
# para que `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from src import config
from src.utils.io import configurar_estilo

configurar_estilo()
print(f"Projeto em {RAIZ}")

In [ ]:
from src.model import perfilagem as perf
from src.model import personas as pers
from src.utils.io import ler_json

segmentada = pd.read_parquet(config.SEGMENTOS)
metadados = ler_json(config.MODELS / "metadados_segmentacao.json")
rotulos = segmentada["segmento"].to_numpy()
pd.Series({k: v for k, v in metadados.items() if not isinstance(v, (dict, list))}).to_frame("valor")

## 5.1 · Confronto com os critérios de aceite

Critério a critério, não em bloco. "O projeto foi bem" não é avaliação.

In [ ]:
from src.pipeline.avaliacao import confrontar_criterios

criterios = confrontar_criterios(segmentada, metadados)
criterios[["criterio", "meta", "obtido", "situacao"]]

## 5.2 · Quanto vale cada segmento

A primeira das duas perguntas que justificam abrir o rótulo reservado.

In [ ]:
leitura = perf.leitura_a_posteriori(segmentada, rotulos)
leitura.insert(0, "persona", [pers.PERSONAS[int(s)]["nome"] for s in leitura.index])
leitura

**S4 (Novo de alto padrão)** concentra 35,6% das unidades e **49,5% da
receita** — responde à segunda pergunta SMART.

Repare também no preço por pé²: **S2 (Compacto de entrada)** tem US$ 116/pé², acima de
**S3 (Familiar consolidado)**, com US$ 107. Imóvel pequeno tem preço unitário maior — o
que muda a conversa sobre margem por metro construído.

## A segmentação é uma tabela de preço disfarçada?

A segunda pergunta que justifica abrir o rótulo. Se os intervalos interquartis de
preço não se sobrepusessem, a resposta seria sim — e a diretoria pediu outra coisa.

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

faixa_preco = pd.qcut(segmentada["preco_venda"], 4, labels=False)
print(f"ARI  segmentos × quartis de preço: {adjusted_rand_score(rotulos, faixa_preco):.4f}")
print(f"NMI  segmentos × quartis de preço: {normalized_mutual_info_score(rotulos, faixa_preco):.4f}")

In [ ]:
perf.sobreposicao_de_preco(segmentada, rotulos)

ARI de 0,31: há relação — imóvel melhor custa mais, obviamente —, mas está
longe de identidade. A segmentação carrega informação de **produto** que o preço
sozinho não tem.

## Validação externa: o bairro

`bairro` ficou fora da modelagem justamente para poder ser usado aqui. Se os segmentos
se distribuírem de forma diferente entre bairros, é sinal de que capturaram estrutura
real de mercado — e não ruído.

In [ ]:
bairros = perf.validacao_por_bairro(segmentada, rotulos, top=8)
bairros.columns = [pers.codigo(int(c)) for c in bairros.columns]
bairros

## PCA × LDA — a distinção que a disciplina trata como eliminatória

O PCA não viu preço nenhum. O LDA foi treinado para separar faixas de preço. Se as
duas projeções contassem a mesma história, a segmentação seria uma tabela de preço.

In [ ]:
from src.pipeline.avaliacao import pca_versus_lda

pca_versus_lda(segmentada)

## 5.3 · Revisão do processo, limitações e próximas etapas

A retrospectiva vale mesmo quando tudo deu certo. E o que não funcionou aparece no
relatório.

In [ ]:
from src.pipeline.avaliacao import limitacoes, proximas_etapas, revisar_processo

revisar_processo()

In [ ]:
limitacoes()

In [ ]:
proximas_etapas()

## 6 · Implantação

Uma segmentação que só existe porque alguém rodou o notebook uma vez **não está
implantada**.

In [ ]:
from src.pipeline.implantacao import executar as implantar

resumo = implantar()
pd.Series(resumo).to_frame("valor")

### O problema de monitorar um modelo sem gabarito

Não existe erro de previsão para acompanhar. O que se acompanha é:

1. a **entrada** mudar de distribuição — *data drift*, medido por PSI;
2. os **grupos** perderem coesão — *cluster drift*, medido pela silhueta no período;
3. a **composição** da carteira se deslocar entre segmentos.

O teste abaixo é honesto: usa o próprio calendário da base, com 2006–2008 como
referência e 2009–2010 como acompanhamento.

In [ ]:
from src.pipeline.implantacao import coesao_no_periodo, medir_drift

drift, composicao = medir_drift(segmentada)
drift.head(8)

In [ ]:
composicao

In [ ]:
coesao_no_periodo(segmentada)

Nenhum atributo em nível de ação; um em atenção (idade do imóvel, PSI 0,10,
efeito natural da passagem do tempo). A composição varia menos de 1 ponto percentual
em todos os segmentos, e a silhueta cai de 0,1797 para 0,1737 — dentro da faixa
aceitável.

**Veredicto: estável.** A segmentação não precisa de reciclagem no horizonte da base.

## O artefato, e a conferência de que ele funciona

O pipeline serializado carrega o pré-processador e o K-Means juntos. A aplicação nunca
reimplementa a preparação — reimplementar a padronização na camada de aplicação é a
forma mais comum de um modelo funcionar no notebook e falhar em produção.

In [ ]:
from src.deployment.scoring import classificar

exemplos = {
    "Casa pequena e antiga": dict(
        area_habitavel=1000, area_porao=800, area_garagem=280, idade_imovel=55,
        anos_desde_reforma=45, qualidade_geral=5, comodos=5, banheiros_total=1.5,
        lareiras=0, acabamento_garagem=1,
    ),
    "Sobrado novo e amplo": dict(
        area_habitavel=2100, area_porao=1400, area_garagem=620, idade_imovel=3,
        anos_desde_reforma=3, qualidade_geral=8, comodos=8, banheiros_total=3.5,
        tem_segundo_pavimento=1, qualidade_cozinha=4, qualidade_externa=4,
        acabamento_garagem=3,
    ),
    "Sem porão, antigo": dict(
        area_habitavel=1150, area_porao=0, area_garagem=400, idade_imovel=55,
        anos_desde_reforma=52, qualidade_geral=4, dormitorios=2,
    ),
}
pd.DataFrame(
    [
        {
            "imóvel": nome,
            "segmento": (r := classificar(atributos))["codigo"],
            "persona": r["persona"],
            "confiança": r["confianca"],
            "razão de fronteira": r["razao_fronteira"],
        }
        for nome, atributos in exemplos.items()
    ]
).set_index("imóvel")

## O ciclo fechado

```
uv run invoke pipeline     # as seis fases, do download aos artefatos
uv run invoke app          # a aplicação que a operação usa
uv run invoke docs         # o relatório publicado
```

A aplicação Streamlit em `src/deployment/app.py` é a entrega da fase 6: sete abas,
uma por pergunta da diretoria, cada uma apontando o CSV que a sustenta.